In [ ]:
import pandas as pd
import numpy as np

# Week 6

df = pd.read_csv("sold_week45.csv")

In [3]:
# dates
date_cols = [
    "ListingContractDate",
    "PurchaseContractDate",
    "CloseDate"
]


print(df.columns)
print(df[date_cols].dtypes)

Index(['Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice',
       'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName',
       'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress',
       'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'AssociationFeeFrequency', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName',
       'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'City',
       'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChangeDate',
       'PurchaseContractDate', 'ListingContractDate', 'StateOrProvince',
       'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighS

In [5]:


for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Price Ratio
df["price_ratio"] = np.where(
    df["OriginalListPrice"] > 0,
    df["ClosePrice"] / df["OriginalListPrice"],
    np.nan
)

# Same
df["close_to_original_list_ratio"] = df["price_ratio"]

# Price Per Square Foot
df["price_per_sqft"] = np.where(
    df["LivingArea"] > 0,
    df["ClosePrice"] / df["LivingArea"],
    np.nan
)

In [6]:
# Time
df["Year"] = df["CloseDate"].dt.year
df["Month"] = df["CloseDate"].dt.month
df["YrMo"] = df["CloseDate"].dt.to_period("M").astype(str)

# Time Difference Features

df["listing_to_contract_days"] = (
    df["PurchaseContractDate"] -
    df["ListingContractDate"]
).dt.days

df["contract_to_close_days"] = (
    df["CloseDate"] -
    df["PurchaseContractDate"]
).dt.days

# Days on Market
df["days_on_market"] = df["DaysOnMarket"]


In [7]:
# Preview
new_cols = [
    "ClosePrice",
    "OriginalListPrice",
    "LivingArea",
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "Year",
    "Month",
    "YrMo",
    "listing_to_contract_days",
    "contract_to_close_days"
]

print("\nSample Output:")
print(df[new_cols].head(10))


Sample Output:
   ClosePrice  OriginalListPrice  LivingArea  price_ratio  \
0    240000.0           499000.0      1140.0     0.480962   
1    815000.0           759900.0      1974.0     1.072510   
2    810000.0           739900.0      1974.0     1.094743   
3   1130000.0           999000.0      2136.0     1.131131   
4   1060000.0          1050000.0      1917.0     1.009524   
5    650000.0           650000.0      1400.0     1.000000   
6   9635000.0         10750000.0      5410.0     0.896279   
7    762000.0           900000.0      1570.0     0.846667   
8   1500000.0          1500000.0      1930.0     1.000000   
9   4825000.0          4825000.0      4700.0     1.000000   

   close_to_original_list_ratio  price_per_sqft  days_on_market  Year  Month  \
0                      0.480962      210.526316             777  2024      1   
1                      1.072510      412.867275              33  2024      1   
2                      1.094743      410.334347             228  2024   

In [8]:
# Segment Analysis Example: County
county_summary = (
    df.groupby("CountyOrParish")
      .agg(
          HomesSold=("ClosePrice", "count"),
          MedianPrice=("ClosePrice", "median"),
          AveragePrice=("ClosePrice", "mean"),
          MedianPPSF=("price_per_sqft", "median"),
          AvgDOM=("days_on_market", "mean"),
          AvgPriceRatio=("price_ratio", "mean")
      )
      .sort_values("MedianPrice", ascending=False)
)

print("\nCounty Summary")
print(county_summary)

property_summary = (
    df.groupby("PropertyType")
      .agg(
          Homes=("ClosePrice", "count"),
          MedianPrice=("ClosePrice", "median"),
          AvgPPSF=("price_per_sqft", "mean"),
          AvgDOM=("days_on_market", "mean")
      )
)

print("\nProperty Type Summary")
print(property_summary)


County Summary
                 HomesSold  MedianPrice  AveragePrice   MedianPPSF  \
CountyOrParish                                                       
Del Norte                1    2485000.0  2.485000e+06   390.907661   
Other County             4    2462500.0  9.504750e+07   242.673993   
San Mateo             6788    1689500.0  2.168499e+06  1045.454545   
Santa Clara          17467    1590000.0  1.905484e+06   962.399284   
Santa Cruz            2851    1200000.0  1.348343e+06   735.135135   
...                    ...          ...           ...          ...   
Lake                  1718     310000.0  3.385131e+05   213.529408   
Imperial               277     306000.0  3.108787e+05   210.862620   
Sierra                   1     255000.0  2.550000e+05   222.902098   
Foreign Country          4     225000.0  2.087500e+05   124.434692   
Lassen                  13     135000.0  2.266538e+05   139.649507   

                     AvgDOM  AvgPriceRatio  
CountyOrParish              

In [9]:
# Save outputs
df.to_csv("week6_sold_feature_engineered.csv", index=False)
county_summary.to_csv("week6_county_summary.csv")
property_summary.to_csv("week6_propertytype_summary.csv")
